# 🔴 Solution: FSDP Training Step

Simulated FSDP: all-gather → forward/backward → reduce-scatter → update

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def fsdp_step(param_shards, grad_fn, world_size):
    """
    简化版 FSDP (Fully Sharded Data Parallel) 训练步骤。

    核心思想：将模型参数分片到各 worker 以减少单设备内存。
    前向传播前 all-gather 完整参数，反向传播后 reduce-scatter 梯度。

    参数:
        param_shards: world_size 个参数分片的列表，每个是扁平 1D 张量
        grad_fn: 接收完整参数张量，返回对应梯度的可调用对象
        world_size: 虚拟 worker 数量

    返回: 更新后的参数分片列表 (lr=0.01)
    """
    # ---- Step 1: All-Gather ----
    # 每个 worker 持有一部分参数，all-gather 拼接得到完整参数
    # 在真实分布式中，这一步是 all-gather 通信
    # 拼接后 shape: (world_size * shard_size,)
    full_param = torch.cat(param_shards)

    # ---- Step 2: 计算梯度 ----
    # 用完整参数计算梯度（等价于单 GPU 训练）
    # grad 的 shape 与 full_param 相同
    grad = grad_fn(full_param)

    # ---- Step 3: Reduce-Scatter ----
    # 将梯度分成 world_size 块，每个 worker 保留自己的那一块
    # 在真实分布式中，这一步是 reduce-scatter 通信
    # 先 reduce (求和梯度，这里只有一份所以不用求和)
    # 再 scatter (分发给各 worker)
    grad_shards = list(grad.chunk(world_size))

    # ---- Step 4: 更新每个分片 ----
    # SGD: param = param - lr * grad
    # 每个 worker 只更新自己持有的分片
    new_shards = [
        param_shards[i] - 0.01 * grad_shards[i]
        for i in range(world_size)
    ]

    return new_shards

In [ ]:
# Verify: one FSDP step should match equivalent SGD on the full parameter
torch.manual_seed(0)
world_size = 4
shard_size = 8

# Initial shards
param_shards = [torch.randn(shard_size) for _ in range(world_size)]

# Simple quadratic loss gradient: grad = 2 * param
grad_fn = lambda p: 2.0 * p

# FSDP step
new_shards = fsdp_step(param_shards, grad_fn, world_size)
fsdp_result = torch.cat(new_shards)

# Reference: plain SGD on full param
full_param = torch.cat(param_shards)
ref_result = full_param - 0.01 * grad_fn(full_param)

print("FSDP result shape:", fsdp_result.shape)  # expect (32,)
print("Max diff vs SGD reference:", (fsdp_result - ref_result).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("fsdp_step")

## 📝 核心思路总结

1. **FSDP 的核心**：参数完全分片存储，前向前 all-gather 拼接，反向后 reduce-scatter 分发梯度。
2. **All-Gather + Reduce-Scatter**：这两个集合通信操作是 FSDP 的灵魂，分别对应前向和反向的参数/梯度需求。
3. **与 DDP 的区别**：DDP 每个 worker 存完整参数（只分片梯度），FSDP 连参数都分片，内存效率更高。
4. **分片更新**：每个 worker 只更新自己持有的参数分片，更新后的分片列表就是下一轮的输入。